# 14_Task8_Step3a — Ensemble 2 TCN có sẵn

**Mục tiêu:** trung bình xác suất của `wide45_lownoise` (126 đặc trưng) và `task8_step2_distance` (546 đặc trưng), xem có tốt hơn `task8_step2_distance` (bản mới nhất) không.

- **Không train gì, không tạo checkpoint mới.** Chỉ nạp 2 checkpoint có sẵn trong `Data_321/checkpoints_v2/`.
- Đánh giá trên tập test, xoay giả lập 0/15/30/45/60°. Cả 3 model (step2, wide45_lownoise, ensemble) được chấm trên **cùng một dữ liệu xoay** để so công bằng.
- Chỉ số theo Decision: tỉ lệ video có khung vừa tự tin ≥ ngưỡng vừa đúng (ngưỡng 0.5→0.9), trung bình 5 ngưỡng, kèm thời điểm đoán đúng đầu tiên (ms, tính từ đầu video = chỉ số khung / 120 fps). Có thêm độ chính xác mọi khung, độ chính xác khi tin cậy ≥ 0.7 và độ phủ để so tiếp với cell 6 của notebook 13.
- Cell 7–8: webcam, phím `m` đổi giữa 3 chế độ (ensemble / step2 / wide45_lownoise).

**Chạy lần lượt cell 1 → 8** bằng kernel `train-gpu` (có CUDA và MediaPipe).

In [9]:
#cell 1: import, nạp tập test và dựng đặc trưng 126 như notebook 13 (chỉ cần test nên nhanh hơn)
import math, time, collections
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook không chạy trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
assert (BASE / "manifest.csv").exists(), f"Không thấy {BASE / 'manifest.csv'}"
m = pd.read_csv(BASE / "manifest.csv")
Tmax = int(m.khung.max())
FPS_DATA = 120

def load_base_features(f):
    lm = np.load(BASE / f)["lm"]
    lm = lm - lm[:, :1]                                   # cổ tay làm gốc
    lm = lm / np.linalg.norm(lm[:, 9], axis=1)[:, None, None]  # chuẩn hóa cỡ bàn tay
    pos = lm.reshape(len(lm), 63)
    has = ~np.isnan(pos[:, 0])
    vel = np.vstack([np.zeros((1, 63), np.float32), np.diff(pos, axis=0)])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32), has

te = m[m.tap == "test"].reset_index(drop=True)
X = np.zeros((len(te), Tmax, 126), np.float32)
M = np.zeros((len(te), Tmax), bool)
for i, f in enumerate(te.path):
    x, h = load_base_features(f)
    X[i, :len(x)], M[i, :len(x)] = x, h
Xte = torch.tensor(X, device=dev)
yte = torch.tensor(te.nhan.values, dtype=torch.long, device=dev)
Mte = torch.tensor(M, device=dev)
print("CUDA:", torch.cuda.get_device_name(0), "| test:", tuple(Xte.shape), "| khung có tay:", int(Mte.sum().item()))

CUDA: NVIDIA GeForce RTX 3050 Ti Laptop GPU | test: (33, 355, 126) | khung có tay: 3930


In [10]:
#cell 2: đặc trưng hình học (546), hàm xoay giả lập, định nghĩa TCN (giống notebook 13)
PAIR_I, PAIR_J = torch.triu_indices(21, 21, offset=1, device=dev)

def features_from_pos(pos, hand_mask):
    valid = hand_mask[:, :, None, None]
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    b, T = pos.shape[:2]
    pos_flat = pos.reshape(b, T, 63)
    pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    vel = torch.zeros_like(pos_flat)
    vel[:, 1:] = (pos_flat[:, 1:] - pos_flat[:, :-1]) * pair_valid
    dist = torch.linalg.vector_norm(pos[:, :, PAIR_I, :] - pos[:, :, PAIR_J, :], dim=-1) * hand_mask[:, :, None]
    dist_vel = torch.zeros_like(dist)
    dist_vel[:, 1:] = (dist[:, 1:] - dist[:, :-1]) * pair_valid
    return torch.cat([pos_flat, vel, dist, dist_vel], dim=-1)

def geo_from_base(X, hand_mask):
    return features_from_pos(X[..., :63].reshape(X.shape[0], X.shape[1], 21, 3), hand_mask)

def rotate_base(X, hand_mask, ax_deg, ay_deg, az_deg):
    b, T = X.shape[:2]
    pos = X[..., :63].reshape(b, T, 21, 3).clone()
    valid = hand_mask[:, :, None, None]
    ax, ay, az = [torch.as_tensor(a, dtype=torch.float32, device=X.device) * math.pi / 180 for a in (ax_deg, ay_deg, az_deg)]
    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    pos = torch.stack([x2 * az.cos() - y1 * az.sin(), x2 * az.sin() + y1 * az.cos(), z2], dim=-1)
    pos = torch.where(valid, pos, torch.zeros_like(pos)).reshape(b, T, 63)
    pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    vel = torch.zeros_like(pos)
    vel[:, 1:] = (pos[:, 1:] - pos[:, :-1]) * pair_valid
    return torch.cat([pos, vel], dim=-1)

class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__()
        self.pad = 2 * d
        self.conv = nn.Conv1d(c, c, 3, dilation=d)
        self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, in_features=126, c=64):
        super().__init__()
        self.inp = nn.Linear(in_features, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))
print("Đã định nghĩa đặc trưng, xoay giả lập và TCN.")

Đã định nghĩa đặc trưng, xoay giả lập và TCN.


In [11]:
#cell 3: nạp 2 checkpoint có sẵn (không train, không ghi file) và hàm xác suất cho từng model
CKPT_OLD = BASE / "checkpoints_v2" / "wide45_lownoise" / "TCN.pt"
CKPT_NEW = BASE / "checkpoints_v2" / "task8_step2_distance" / "TCN.pt"
for p in (CKPT_OLD, CKPT_NEW):
    assert p.exists(), f"Không thấy checkpoint: {p}"

def load_tcn(path, in_features):
    ck = torch.load(path, map_location=dev, weights_only=False)
    net = TCN(in_features=in_features).to(dev)
    net.load_state_dict(ck["w"])
    return net.eval(), ck

net_old, ck_old = load_tcn(CKPT_OLD, 126)
net_new, ck_new = load_tcn(CKPT_NEW, 546)

def probs_old(Xb, Mb):            # Xb: (B,T,126) đặc trưng gốc
    return net_old(Xb).softmax(-1)

def probs_new(Xb, Mb):            # tính thêm 420 đặc trưng hình học từ Xb
    return net_new(geo_from_base(Xb, Mb)).softmax(-1)

def probs_ens(Xb, Mb):            # trung bình xác suất (soft voting), trọng số 0.5/0.5
    return (probs_old(Xb, Mb) + probs_new(Xb, Mb)) / 2

print(f"wide45_lownoise: epoch {ck_old.get('epoch', '?')} | step2: epoch {ck_new.get('epoch', '?')}")

wide45_lownoise: epoch 49 | step2: epoch 25


In [12]:
#cell 4: hàm chấm điểm theo tiêu chí Decision (ngưỡng 0.5→0.9) + ms đoán đúng đầu tiên
THRS = [0.5, 0.6, 0.7, 0.8, 0.9]

def metrics(P, y, Mk):
    conf, pred = P.max(-1)                               # (N,T)
    ok = (pred == y[:, None]) & Mk
    n = Mk.sum().item()
    r = {"acc_all": ok.sum().item() / n * 100}
    for t in THRS:
        k = int(round(t * 100))
        hit = ok & (conf >= t)                           # khung vừa tự tin vừa đúng
        has = hit.any(1)                                 # video có ít nhất 1 khung như vậy
        first = hit.float().argmax(1).float()            # chỉ số khung đầu tiên
        r[f"v{k}"] = has.float().mean().item() * 100
        r[f"ms{k}"] = (first[has].mean().item() / FPS_DATA * 1000) if has.any() else float("nan")
    r["tb"] = float(np.mean([r[f"v{int(round(t*100))}"] for t in THRS]))
    sel = (conf >= 0.7) & Mk
    r["accc70"] = (ok & sel).sum().item() / max(sel.sum().item(), 1) * 100
    r["cov70"] = sel.sum().item() / n * 100
    return r
print("Sẵn sàng chấm điểm.")

Sẵn sàng chấm điểm.


In [13]:
#cell 5: chấm 3 model ở góc 0/15/30/45/60 (cùng dữ liệu xoay cho cả 3; góc >0 lặp 5 lần lấy trung bình)
MAGS, REPEAT = [0, 15, 30, 45, 60], 5
NAMES = ["step2 (mới nhất)", "wide45_lownoise", "ensemble 3a"]
g = torch.Generator(device=dev).manual_seed(0)
rows = []
with torch.no_grad():
    for mag in MAGS:
        acc = {n: [] for n in NAMES}
        for _ in range(1 if mag == 0 else REPEAT):
            if mag == 0:
                Xr = Xte
            else:
                ang = lambda: (torch.rand(len(Xte), 1, 1, device=dev, generator=g) * 2 - 1) * mag
                Xr = rotate_base(Xte, Mte, ang(), ang(), ang())
            P_old, P_new = probs_old(Xr, Mte), probs_new(Xr, Mte)
            for n, P in zip(NAMES, [P_new, P_old, (P_old + P_new) / 2]):
                acc[n].append(metrics(P, yte, Mte))
        for n, v in acc.items():
            rows.append({"model": n, "goc": mag, **{k: float(np.nanmean([d[k] for d in v])) for k in v[0]}})
res = pd.DataFrame(rows)

def show(col, title):
    print("\n" + title)
    display(res.pivot_table(index="model", columns="goc", values=col).reindex(NAMES).round(1))

show("tb",     "Tiêu chí Decision: TB 5 ngưỡng của tỉ lệ video (%)")
show("v50",    "Tỉ lệ video ở ngưỡng 0.5 (%)")
show("v60",    "Tỉ lệ video ở ngưỡng 0.6 (%)")
show("acc_all","Độ chính xác mọi khung có tay (%)")
show("accc70", "Độ chính xác khi tin cậy ≥ 0.7 (%)")
show("cov70",  "Độ phủ: % khung đạt ngưỡng 0.7")
show("ms70",   "Thời điểm đoán đúng đầu tiên ở ngưỡng 0.7 (ms, tính từ đầu video)")


Tiêu chí Decision: TB 5 ngưỡng của tỉ lệ video (%)


goc,0,15,30,45,60
model,,,,,
step2 (mới nhất),98.2,98.5,98.7,98.7,98.8
wide45_lownoise,99.4,99.2,98.9,98.4,95.6
ensemble 3a,99.4,99.0,99.0,98.5,97.1



Tỉ lệ video ở ngưỡng 0.5 (%)


goc,0,15,30,45,60
model,,,,,
step2 (mới nhất),100.0,100.0,100.0,100.0,100.0
wide45_lownoise,100.0,100.0,100.0,100.0,98.2
ensemble 3a,100.0,100.0,100.0,100.0,100.0



Tỉ lệ video ở ngưỡng 0.6 (%)


...

goc,0,15,30,45,60
model,,,,,
step2 (mới nhất),95.8,95.9,95.9,95.3,95.6
wide45_lownoise,96.7,96.7,97.2,97.7,96.4
ensemble 3a,96.7,96.6,96.8,96.4,96.3



Độ phủ: % khung đạt ngưỡng 0.7


goc,0,15,30,45,60
model,,,,,
step2 (mới nhất),91.8,91.7,91.7,92.1,91.8
wide45_lownoise,89.4,89.2,87.2,85.9,84.6
ensemble 3a,90.2,90.2,89.2,88.7,87.7



Thời điểm đoán đúng đầu tiên ở ngưỡng 0.7 (ms, tính từ đầu video)


goc,0,15,30,45,60
model,,,,,
step2 (mới nhất),116.4,116.1,116.8,117.5,114.4
wide45_lownoise,129.0,129.9,136.9,142.7,140.4
ensemble 3a,122.0,121.9,125.1,131.9,122.4


In [14]:
#cell 6: chênh lệch ensemble so với step2 (bản mới nhất) và kết luận sơ bộ
p = res.pivot_table(index="model", columns="goc", values=["tb", "v50", "v60", "acc_all", "accc70", "cov70"])
for col in ["tb", "v50", "v60", "acc_all", "accc70", "cov70"]:
    d = p[col].loc["ensemble 3a"] - p[col].loc["step2 (mới nhất)"]
    print(f"Δ {col:8s} (ensemble - step2): " + " | ".join(f"{int(a):2d}°: {v:+.1f}" for a, v in d.items()))

d_acc = p["acc_all"].loc["ensemble 3a"] - p["acc_all"].loc["step2 (mới nhất)"]
if d_acc[0] >= -1 and d_acc[45] > 0:
    print("\nKẾT QUẢ SỐ LIỆU: ĐẠT tiêu chí sơ bộ (không giảm quá 1 điểm ở 0°, tăng ở ±45°). Cần xác nhận bằng webcam.")
else:
    print("\nKẾT QUẢ SỐ LIỆU: CHƯA ĐẠT tiêu chí sơ bộ; giữ step2, cân nhắc 3b.")
print("Lưu ý: tiêu chí chỉ để tham khảo; test chỉ 33 video nên chênh dưới ~1 điểm chưa chắc chắn.")

Δ tb       (ensemble - step2):  0°: +1.2 | 15°: +0.5 | 30°: +0.4 | 45°: -0.1 | 60°: -1.7
Δ v50      (ensemble - step2):  0°: +0.0 | 15°: +0.0 | 30°: +0.0 | 45°: +0.0 | 60°: +0.0
Δ v60      (ensemble - step2):  0°: +0.0 | 15°: +0.0 | 30°: +0.0 | 45°: +0.0 | 60°: -0.6
Δ acc_all  (ensemble - step2):  0°: -0.9 | 15°: -0.4 | 30°: -0.7 | 45°: -0.4 | 60°: -0.1
Δ accc70   (ensemble - step2):  0°: +0.9 | 15°: +0.7 | 30°: +0.9 | 45°: +1.2 | 60°: +0.6
Δ cov70    (ensemble - step2):  0°: -1.7 | 15°: -1.5 | 30°: -2.4 | 45°: -3.4 | 60°: -4.1

KẾT QUẢ SỐ LIỆU: CHƯA ĐẠT tiêu chí sơ bộ; giữ step2, cân nhắc 3b.
Lưu ý: tiêu chí chỉ để tham khảo; test chỉ 33 video nên chênh dưới ~1 điểm chưa chắc chắn.


In [15]:
#cell 7: chuẩn bị MediaPipe + dự đoán webcam cho 3 chế độ (giao diện giống 08_WebcamWide / notebook 13)
import cv2
import mediapipe as mp

TASK = BASE / "hand_landmarker.task"
assert TASK.exists(), f"Không thấy {TASK}"
CLASSES = ["paper", "rock", "scissors"]
COLORS = {0: (60, 60, 255), 1: (255, 150, 40), 2: (60, 210, 60)}
GRAY, BLACK = (190, 190, 190), (0, 0, 0)
BONES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
         (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]
MODES = ["ens", "new", "old"]
MODE_NAME = {"ens": "ENSEMBLE 3a", "new": "STEP 2", "old": "WIDE45_LOWNOISE"}

landmarker = mp.tasks.vision.HandLandmarker.create_from_options(
    mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO, num_hands=1))
T0 = time.perf_counter()

def detect(frame_bgr, ts_ms):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    result = landmarker.detect_for_video(mp.Image(mp.ImageFormat.SRGB, rgb), ts_ms)
    if not result.hand_landmarks:
        return None
    return np.array([[p.x, p.y, p.z] for p in result.hand_landmarks[0]], np.float32)

FPS_MODEL, WIN = 120, 128
buf = collections.deque(maxlen=WIN)
state = {"prev_lm": None, "prev_t": None}

def push(lm, t):
    prev, pt = state["prev_lm"], state["prev_t"]
    n = 1 if pt is None else max(1, round((t - pt) * FPS_MODEL))
    if lm is None:
        buf.extend([np.full((21, 3), np.nan, np.float32)] * n)
    elif prev is None:
        buf.append(lm)
    else:
        buf.extend(prev + (lm - prev) * (k / n) for k in range(1, n + 1))
    state["prev_lm"], state["prev_t"] = lm, t

def webcam_base_features(lm):
    has = ~np.isnan(lm[:, 0, 0])
    q = lm - lm[:, :1]
    scale = np.linalg.norm(q[:, 9], axis=1)[:, None, None]
    q = np.nan_to_num(q / np.maximum(scale, 1e-8)).astype(np.float32)
    q[~has] = 0
    pos = q.reshape(len(q), 63)
    vel = np.zeros_like(pos)
    if len(pos) > 1:
        vel[1:] = (pos[1:] - pos[:-1]) * (has[1:] & has[:-1])[:, None]
    return np.hstack([pos, vel]).astype(np.float32), has

def predict(which):
    lm = np.stack(buf)
    if np.isnan(lm[-1, 0, 0]):
        return None
    base, has = webcam_base_features(lm)
    xb, mb = torch.from_numpy(base)[None].to(dev), torch.from_numpy(has)[None].to(dev)
    with torch.no_grad():
        fn = {"ens": probs_ens, "new": probs_new, "old": probs_old}[which]
        probs = fn(xb, mb)[0, -1]
    conf, cls = probs.max(0)
    return int(cls.item()), float(conf.item())

def put_label(img, text, x, y, bg, scale=0.75):
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.rectangle(img, (x, y-h-base-6), (x+w+10, y+2), bg, -1)
    cv2.putText(img, text, (x+5, y-base), cv2.FONT_HERSHEY_SIMPLEX, scale, BLACK, 2, cv2.LINE_AA)

def draw_hand(img, landmarks, color, mirror=True):
    h, w = img.shape[:2]
    px, py = (landmarks[:, 0] * w).astype(int), (landmarks[:, 1] * h).astype(int)
    if mirror:
        px = w - 1 - px
    for a, b in BONES:
        cv2.line(img, (px[a], py[a]), (px[b], py[b]), color, 2)
    cv2.rectangle(img, (max(0, px.min()-12), max(0, py.min()-12)),
                  (min(w-1, px.max()+12), min(h-1, py.max()+12)), color, 2)

def draw_hud(img, fps, threshold, which):
    put_label(img, f"FPS {fps:.1f} | threshold {threshold:.2f}", 10, 30, (230, 230, 230), 0.6)
    put_label(img, f"MODEL: {MODE_NAME[which]}", 10, 65, (230, 230, 230), 0.6)
    put_label(img, "m: doi model | [ ]: nguong | q: thoat", 10, 100, (230, 230, 230), 0.55)
print("Đã nạp MediaPipe; webcam chỉ chạy ở cell 8.")

Đã nạp MediaPipe; webcam chỉ chạy ở cell 8.


In [16]:
#cell 8: chạy webcam; m đổi model (ensemble → step2 → wide45_lownoise), [ ] chỉnh ngưỡng, q thoát
CAM_ID, MIRROR, THR_CAM, WHICH = 0, True, 0.70, "ens"
cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
assert cap.isOpened(), "Không mở được webcam; thử đổi CAM_ID = 1."

buf.clear()
state.update(prev_lm=None, prev_t=None)
fps, last = 0.0, time.perf_counter()
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        now = time.perf_counter()
        lm = detect(frame, int((now - T0) * 1000))
        push(lm, now)
        out = predict(WHICH) if lm is not None else None
        disp = cv2.flip(frame, 1) if MIRROR else frame
        fps = 0.9 * fps + 0.1 / max(now - last, 1e-6)
        last = now
        if out is None:
            put_label(disp, "NO HAND", 10, 150, GRAY)
        else:
            cls, conf = out
            sure = conf >= THR_CAM
            color = COLORS[cls] if sure else GRAY
            draw_hand(disp, lm, color, MIRROR)
            put_label(disp, f"{CLASSES[cls]} {conf:.0%}" if sure else f"{CLASSES[cls]}? {conf:.0%}", 10, 150, color)
        draw_hud(disp, fps, THR_CAM, WHICH)
        cv2.imshow("Task 8 Step 3a - RPS webcam", disp)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("m"):
            WHICH = MODES[(MODES.index(WHICH) + 1) % len(MODES)]
        elif key == ord("["):
            THR_CAM = max(0.05, THR_CAM - 0.05)
        elif key == ord("]"):
            THR_CAM = min(0.95, THR_CAM + 0.05)
finally:
    cap.release()
    cv2.destroyAllWindows()
    landmarker.close()
print("Đã đóng webcam.")

Đã đóng webcam.
